In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       3.7535, ∥Δ∥: 0.7383, η: 1.00, ∥Δᶜ∥: 0.7383, accepted: True , log 𝓛:  -5.5326
[  2] ∥∇∥:       4.4534, ∥Δ∥: 0.2132, η: 1.00, ∥Δᶜ∥: 0.2132, accepted: True , log 𝓛:  -4.9983
[  3] ∥∇∥:       0.8228, ∥Δ∥: 0.0617, η: 1.00, ∥Δᶜ∥: 0.0617, accepted: True , log 𝓛:  -4.9724
[  4] ∥∇∥:       0.0469, ∥Δ∥: 0.0040, η: 1.00, ∥Δᶜ∥: 0.0040, accepted: True , log 𝓛:  -4.9723
[  5] ∥∇∥:       0.0001, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -4.9723

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([-0.1648, -0.4480]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-5.5326),
 tensor(-4.9983),
 tensor(-4.9724),
 tensor(-4.9723),
 tensor(-4.9723)]

In [3]:
torch.stack(history["loglik"])

tensor([-5.5326, -4.9983, -4.9724, -4.9723, -4.9723])

In [4]:
torch.stack(history["free_params"])

tensor([[-0.1389, -0.7251],
        [-0.1509, -0.5122],
        [-0.1637, -0.4519],
        [-0.1648, -0.4480],
        [-0.1648, -0.4480]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([-0.1648, -0.4480]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([0.7192]), 'R/sigma^2': tensor([0.4082])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.1274, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192,
          0.0000],
         [0.0000, 1.1274, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000,
          0.7192],
         [0.7192, 0.0000, 1.1274, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192,
          0.0000],
         [0.0000, 0.7192, 0.0000, 1.1274, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000,
          0.7192],
         [0.7192, 0.0000, 0.7192, 0.0000, 1.1274, 0.0000, 0.7192, 0.0000, 0.7192,
          0.0000],
         [0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 1.1274, 0.0000, 0.7192, 0.0000,
          0.7192],
         [0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 1.1274, 0.0000, 0.7192,
          0.0000],
         [0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 1.1274, 0.0000,
          0.7192],
         [0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 1.1274,
          0.0000],
         [0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000, 0.7192, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.4491, -0.4573])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[0.0293, 0.0157],
        [0.0157, 0.0688]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.1711, 0.2623])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.6915, -0.8678])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[0.5889, 0.0032],
        [0.0032, 0.6424]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([0.5889, 0.6424])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.6915, -0.8678])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[0.5889, 0.0032],
         [0.0032, 0.6424]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([0.5889, 0.6424])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([-0.6209,  0.7244,  0.7736,  0.5274,  1.0974,  1.3518,  0.7843, -1.2331,
         0.5651, -0.4491])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 0.0706, -0.1434,  1.4651, -0.3404,  1.7889,  0.4840,  1.4757, -2.1009,
         1.2565, -1.3169])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 0.0706, -0.1434])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 0.0706, -0.1434])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.5868, -0.7943,  0.6020, -0.8039, -0.2332, -0.5604,  1.2735, -1.2502,
         0.6207, -1.4229])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.8954,  0.0735, -0.0895,  0.0640, -0.9246,  0.3074,  0.5820, -0.3824,
        -0.0707, -0.5551])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[0.0293, 0.0157],
         [0.0157, 0.0688]]),
 tensor([[ 0.0119, -0.0101],
         [ 0.0625, -0.0044]]),
 tensor([[ 0.0119,  0.0625],
         [-0.0101, -0.0044]]),
 tensor([[ 0.1303, -0.0032],
         [-0.0032,  0.0768]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-4.9723), 'cform')

In [25]:
reml.is_converged()

True